# 01B. Signal Family Expansion Plan

Purpose: define a clean research-design layer before Notebook 02 Signal Factory. This notebook documents and structures the next signal-family expansion roadmap, but it does not generate signals or modify formulas.

Scope boundaries:
- Planning and documentation only.
- Writes a structured plan to SQLite.
- Does not modify Notebook 02, `src/signals.py`, signal metadata logic, scoring, WFV, alpha construction, stress, freeze, portfolio, or ML logic.


## 1. Purpose and Scope

Notebook 03G showed that the current research-approved signal set is highly redundant. Notebook 01B creates the research brief for candidate expansion so the next Signal Factory upgrades can target genuinely complementary families rather than simply adding more variants of the same volatility behavior.

This update adds a second targeted expansion wave after the latest 03E/03G evidence showed the approved pool still clustering around volatility, defensive, and liquidity-like behavior. The notebook remains planning-only: it writes a structured roadmap to SQLite and does not generate signals.

## 2. Current Problem Summary

- Notebook 03G found `effective_signal_count` around `1.12` across the current approved alpha-research signal candidates.
- The approved set is mostly volatility / defensive-volatility behavior, with limited independent signal breadth.
- Several approved candidates have near-perfect correlation after direction adjustment.
- The current alpha pool therefore lacks true family diversity, even though the broader raw candidate library contains many formulas.
- Wave 1 targeted immediate high-priority additions; the second wave adds a narrower plan aimed at breaking the remaining volatility / defensive / liquidity cluster before the next controlled Signal Factory expansion.

## 3. Imports and Config

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "2-Phase 2_Signal Expansion":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.db import get_db_path, load_table, table_exists
from src.run_config import make_run_id, make_run_timestamp
from src.signal_family_plan import (
    SIGNAL_FAMILY_PLAN_TABLES,
    build_signal_family_expansion_plan,
    save_signal_family_expansion_plan,
)
from src.signals import PHASE2_SIGNAL_SPECS

DB_PATH = get_db_path()
SIGNAL_FAMILY_PLAN_VERSION = "phase2_signal_family_plan_v1"

pd.set_option("display.max_columns", 200)
DB_PATH

PosixPath('/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db')

## 4. Create run_id / timestamp

In [2]:
run_id = make_run_id(prefix="phase2_signal_family_plan")
run_timestamp = make_run_timestamp()

run_id, run_timestamp

('phase2_signal_family_plan_20260502_215958', '2026-05-02 21:59:58')

## 5. Current Signal Families Already Present

In [3]:
current_signal_specs = pd.DataFrame(PHASE2_SIGNAL_SPECS)
current_family_summary = (
    current_signal_specs.groupby("signal_family", dropna=False)
    .agg(
        n_signals=("signal_name", "nunique"),
        example_signals=("signal_name", lambda values: ", ".join(sorted(values.astype(str).head(5)))),
    )
    .reset_index()
    .sort_values(["n_signals", "signal_family"], ascending=[False, True])
)

if table_exists("signal_health_score_current", db_path=DB_PATH):
    health_score = load_table("signal_health_score_current", db_path=DB_PATH)
    approved_health = health_score.loc[
        health_score.get("signal_health_gate", pd.Series(index=health_score.index, dtype=object)).eq("APPROVED_FOR_RESEARCH")
    ].copy()
    health_family_summary = (
        approved_health.groupby("signal_family", dropna=False)
        .agg(
            n_approved=("signal_name", "nunique"),
            avg_health_score=("signal_health_score", "mean"),
            max_health_score=("signal_health_score", "max"),
        )
        .reset_index()
        .sort_values(["n_approved", "max_health_score"], ascending=[False, False])
    )
else:
    health_score = pd.DataFrame()
    health_family_summary = pd.DataFrame()

if table_exists("signal_diversity_diagnostics_current", db_path=DB_PATH):
    diversity_diagnostics = load_table("signal_diversity_diagnostics_current", db_path=DB_PATH)
else:
    diversity_diagnostics = pd.DataFrame()

if table_exists("signal_diversity_selection_current", db_path=DB_PATH):
    diversity_selection = load_table("signal_diversity_selection_current", db_path=DB_PATH)
else:
    diversity_selection = pd.DataFrame()

print(f"Current signal specs: {len(current_signal_specs)}")
display(current_family_summary)

print("Latest 03E approved health family summary")
display(health_family_summary)

print("Latest 03G diversity diagnostics")
display(diversity_diagnostics)

print("Latest 03G selected/redundant signals")
display(diversity_selection)

Current signal specs: 50


,signal_family,n_signals,example_signals
10,trend_quality,7,"ma_slope_50, price_above_ma_100, price_above_m..."
0,breakout,6,"breakout_20, breakout_60, breakout_up_20, brea..."
9,short_term_reversal,6,"distance_from_ma_20, gap_reversal_1d, reversal..."
4,defensive_stability,4,"downside_vol_20, downside_vol_60, return_stabi..."
15,volume_liquidity,4,"dollar_volume_20, liquidity_rank_20, volume_tr..."
2,cross_sectional_relative_strength,3,"relative_strength_120, relative_strength_20, r..."
3,defensive_quality,3,"low_vol_strength, risk_adjusted_momentum_20, r..."
6,mean_reversion,3,"intraday_reversal_1d, mean_reversion_20, mean_..."
8,residual_momentum,3,"residual_momentum_120, residual_momentum_20, r..."
7,momentum,2,"momentum_20, momentum_60"


Latest 03E approved health family summary


,signal_family,n_approved,avg_health_score,max_health_score
3,volatility,2,79.75,86.0
2,liquidity,1,71.50,73.0
1,defensive_stability,1,72.00,72.0
0,defensive_quality,1,70.00,70.0


Latest 03G diversity diagnostics


,n_candidates,avg_abs_correlation,max_abs_correlation,median_abs_correlation,effective_signal_count,run_id,diversity_version
0,5,0.929185,1.0,1.0,1.115169,phase2_signal_diversity_20260502_170722,phase2_signal_diversity_v1


Latest 03G selected/redundant signals


,signal_key,signal_name,horizon,signal_family,signal_health_score,final_research_gate,reproducibility_status,pass_rate,avg_effective_mean_ic,selected_flag,selection_rank,max_corr_to_selected,selection_reason,diversity_group,run_id,diversity_version
0,volatility_20__h10,volatility_20,10,volatility,86.0,APPROVED_FOR_ALPHA_RESEARCH,GLOBAL_PASS,0.928571,0.044357,1,1.0,0.000000,Selected within correlation threshold 0.85.,CORE_SELECTED,phase2_signal_diversity_20260502_170722,phase2_signal_diversity_v1
1,volatility_60__h5,volatility_60,5,volatility,72.0,APPROVED_FOR_ALPHA_RESEARCH,CONDITIONAL_PASS,0.857143,0.034193,1,2.0,0.822962,Selected within correlation threshold 0.85.,CORE_SELECTED,phase2_signal_diversity_20260502_170722,phase2_signal_diversity_v1
2,volatility_20__h5,volatility_20,5,volatility,83.0,APPROVED_FOR_ALPHA_RESEARCH,GLOBAL_PASS,1.000000,0.034930,1,3.0,1.000000,Forced to meet MIN_SELECTED after threshold re...,FORCED_MIN_SELECTED,phase2_signal_diversity_20260502_170722,phase2_signal_diversity_v1
3,volatility_20__h20,volatility_20,20,volatility,78.0,APPROVED_FOR_ALPHA_RESEARCH,GLOBAL_PASS,0.928571,0.057128,0,NaN,1.000000,Rejected as redundant; max abs correlation 1.0...,REDUNDANT_REJECTED,phase2_signal_diversity_20260502_170722,phase2_signal_diversity_v1
4,low_vol_strength__h10,low_vol_strength,10,defensive_quality,70.0,APPROVED_FOR_ALPHA_RESEARCH,GLOBAL_PASS,0.928571,0.044357,0,NaN,1.000000,Rejected as redundant; max abs correlation 1.0...,REDUNDANT_REJECTED,phase2_signal_diversity_20260502_170722,phase2_signal_diversity_v1


## 6. Target Signal Families to Expand

### mean_reversion / short_term_reversal
Economic intuition: forced selling, overnight gaps, and short-lived price pressure can reverse when liquidity returns. These should be most useful at 1-10 day horizons and should not require a persistent high-volatility regime.

### volume_flow
Economic intuition: volume can reveal participation, accumulation, distribution, and attention. Flow-style signals can be lowly correlated to volatility when they condition price moves on participation rather than raw price variance.

### liquidity
Economic intuition: liquidity changes and price impact can affect forward returns through compensation, crowding, and tradability. Liquidity candidates should help separate volatile-but-liquid from volatile-and-hard-to-trade names.

### correlation_dispersion
Economic intuition: market beta, rolling correlation, and idiosyncratic return can reveal changing systematic exposure or firm-specific behavior. This targets structure inside volatility rather than simple volatility level.

### volatility_change / volatility_shock
Economic intuition: fresh changes in volatility may behave differently from persistent volatility. These are related to volatility, so they should be tested carefully for redundancy in 03G.

### selective momentum / trend
Economic intuition: smoother, participation-supported trends may work outside crisis regimes and should diversify away from high-drawdown volatility survivors.

### cross_sectional_relative_value
Economic intuition: relative returns and relative risk positioning can separate stock-specific opportunity from broad market regime behavior, giving the next expansion a way to test cross-sectional edges that are not merely high-volatility rankings.

## 7. Structured Signal-Family Expansion Plan

In [4]:
signal_family_expansion_plan = build_signal_family_expansion_plan()

wave_counts = (
    signal_family_expansion_plan["expansion_wave"]
    .value_counts(dropna=False)
    .rename_axis("expansion_wave")
    .reset_index(name="n_proposed_signals")
)
implementation_status_counts = (
    signal_family_expansion_plan["implementation_status"]
    .value_counts(dropna=False)
    .rename_axis("implementation_status")
    .reset_index(name="n_proposed_signals")
)
priority_counts = (
    signal_family_expansion_plan["implementation_priority"]
    .value_counts(dropna=False)
    .rename_axis("implementation_priority")
    .reset_index(name="n_proposed_signals")
)
family_counts = (
    signal_family_expansion_plan["family_name"]
    .value_counts(dropna=False)
    .rename_axis("family_name")
    .reset_index(name="n_proposed_signals")
)

print("Expansion wave counts")
display(wave_counts)

print("Implementation status counts")
display(implementation_status_counts)

print("Priority counts")
display(priority_counts)

print("Family counts")
display(family_counts)

display(signal_family_expansion_plan)

Expansion wave counts


,expansion_wave,n_proposed_signals
0,wave_1,12
1,wave_2,8


Implementation status counts


,implementation_status,n_proposed_signals
0,planned,15
1,implemented,5


Priority counts


,implementation_priority,n_proposed_signals
0,HIGH,10
1,MEDIUM,9
2,LOW,1


Family counts


,family_name,n_proposed_signals
0,volume_flow,4
1,correlation_dispersion,4
2,liquidity,2
3,volatility_change,2
4,selective_momentum,2
5,cross_sectional_relative_value,2
6,mean_reversion,1
7,short_term_reversal,1
8,volatility_shock,1
9,selective_trend,1


,expansion_wave,family_name,proposed_signal_name,implementation_status,formula_description,required_inputs,expected_horizon,expected_diversification_role,implementation_priority,implementation_batch,reason_added,economic_intuition,expected_correlation_behavior_vs_volatility,risk_of_redundancy,notes
0,wave_1,mean_reversion,intraday_reversal_1d,implemented,"Negative same-day open-to-close return, ranked...","open,close",1-5d,Adds very short-horizon pullback behavior dist...,HIGH,already_implemented,Initial diversity expansion roadmap from 01B a...,Short-term overreaction and liquidity-demand s...,Low to moderate; may overlap during selloffs b...,MEDIUM,Keep simple and avoid using future intraday in...
1,wave_1,short_term_reversal,gap_reversal_1d,implemented,Negative overnight gap from prior close to cur...,"open,close",1-5d,Separates overnight dislocation reversal from ...,HIGH,already_implemented,Initial diversity expansion roadmap from 01B a...,Large overnight repricing can overshoot before...,"Moderate in stress, lower in normal regimes.",MEDIUM,"Use only current open and previous close, with..."
2,wave_1,volume_flow,up_down_volume_pressure_20,implemented,Rolling difference between volume on up days a...,"close,volume",5-20d,Adds participation/flow information not captur...,HIGH,already_implemented,Initial diversity expansion roadmap from 01B a...,Persistent accumulation or distribution can re...,Low to moderate; flow pressure can be positive...,LOW,Do not dollar-weight by future prices; use con...
3,wave_1,volume_flow,volume_price_confirmation_20,planned,20-day return multiplied by rolling volume z-s...,"close,volume",10-20d,Tests whether moves supported by unusual volum...,MEDIUM,later_or_low,Initial diversity expansion roadmap from 01B a...,Price moves with participation may be more dur...,Moderate; volume spikes can accompany volatili...,MEDIUM,Clip or rank to reduce domination by single ex...
4,wave_1,liquidity,amihud_illiq_20,implemented,Rolling mean of abs daily return divided by do...,"close,volume",5-20d,Adds tradability/liquidity-premium behavior di...,HIGH,already_implemented,Initial diversity expansion roadmap from 01B a...,Illiquid names may require compensation or exp...,Moderate; related to volatility but scaled by ...,MEDIUM,Handle zero dollar volume safely and leave mis...
5,wave_1,liquidity,dollar_volume_change_20,planned,20-day rolling dollar volume divided by 60-day...,"close,volume",5-20d,Captures changing investor attention and liqui...,LOW,later_or_low,Initial diversity expansion roadmap from 01B a...,Rising trading activity can precede repricing ...,Low to moderate; activity changes need not imp...,LOW,Use trailing averages only.
6,wave_1,correlation_dispersion,market_beta_change_60,implemented,Rolling 20-day beta to benchmark minus rolling...,"close,benchmark_close",10-20d,Identifies changing market sensitivity rather ...,HIGH,already_implemented,Initial diversity expansion roadmap from 01B a...,"Rapid beta changes can reveal crowding, de-ris...",Moderate; beta instability may rise in stress ...,MEDIUM,Use benchmark returns from existing clean benc...
7,wave_1,correlation_dispersion,idiosyncratic_vol_ratio_20_60,planned,20-day residual volatility to benchmark divide...,"close,benchmark_close",5-20d,Separates idiosyncratic shock behavior from br...,MEDIUM,later_or_low,Initial diversity expansion roadmap from 01B a...,Firm-specific risk shocks can mean-revert or p...,"Moderate to high in stress, but more stock-spe...",MEDIUM_HIGH,Treat as diagnostic candidate; 03G should deci...
8,wave_1,volatility_change,volatility_shock_5_20,planned,5-day realized volatility divided by 20-day re...,close,1-10d,Captures volatility acceleration rather than p...,MEDIUM,later_or_low,Initial diversity expansion roadmap from 01B a...,Fresh volatility shocks may behave differently...,Moderate; intentionally related but less level...,MEDIUM_HIGH,Include only if 03G shows it is not a clone of...
9,wave_1,volatility_shock,range

## 8. Save Plan to SQLite

In [5]:
saved_paths = save_signal_family_expansion_plan(
    plan=signal_family_expansion_plan,
    db_path=DB_PATH,
    run_id=run_id,
    plan_version=SIGNAL_FAMILY_PLAN_VERSION,
    timestamp=run_timestamp,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            "artifact": artifact,
            "current_table": tables[0],
            "history_table": tables[1],
            "sqlite_path": str(saved_paths[artifact]),
        }
        for artifact, tables in SIGNAL_FAMILY_PLAN_TABLES.items()
    ]
)

display(sqlite_tables_written)

,artifact,current_table,history_table,sqlite_path
0,plan,signal_family_expansion_plan_current,signal_family_expansion_plan_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## 9. Final Summary

In [6]:
final_summary = pd.DataFrame(
    [
        {"metric": "run_id", "value": run_id},
        {"metric": "run_timestamp", "value": run_timestamp},
        {"metric": "signal_family_plan_version", "value": SIGNAL_FAMILY_PLAN_VERSION},
        {"metric": "current_signal_families", "value": current_family_summary["signal_family"].nunique()},
        {"metric": "proposed_signal_families", "value": signal_family_expansion_plan["family_name"].nunique()},
        {"metric": "proposed_signal_count", "value": len(signal_family_expansion_plan)},
        {"metric": "expansion_waves", "value": signal_family_expansion_plan["expansion_wave"].nunique()},
        {"metric": "wave_2_proposed_signal_count", "value": int(signal_family_expansion_plan["expansion_wave"].eq("wave_2").sum())},
        {"metric": "implemented_wave_1_count", "value": int(signal_family_expansion_plan["implementation_status"].eq("implemented").sum())},
    ]
)

print("Current problem summary from 03G")
display(diversity_diagnostics)

print("03E approved health family summary")
display(health_family_summary)

print("Expansion wave counts")
display(wave_counts)

print("Implementation status counts")
display(implementation_status_counts)

print("Target expansion plan")
display(signal_family_expansion_plan)

print("SQLite tables written")
display(sqlite_tables_written)

print("Final summary")
display(final_summary)

Current problem summary from 03G


,n_candidates,avg_abs_correlation,max_abs_correlation,median_abs_correlation,effective_signal_count,run_id,diversity_version
0,5,0.929185,1.0,1.0,1.115169,phase2_signal_diversity_20260502_170722,phase2_signal_diversity_v1


03E approved health family summary


,signal_family,n_approved,avg_health_score,max_health_score
3,volatility,2,79.75,86.0
2,liquidity,1,71.50,73.0
1,defensive_stability,1,72.00,72.0
0,defensive_quality,1,70.00,70.0


Expansion wave counts


...

,expansion_wave,family_name,proposed_signal_name,implementation_status,formula_description,required_inputs,expected_horizon,expected_diversification_role,implementation_priority,implementation_batch,reason_added,economic_intuition,expected_correlation_behavior_vs_volatility,risk_of_redundancy,notes
0,wave_1,mean_reversion,intraday_reversal_1d,implemented,"Negative same-day open-to-close return, ranked...","open,close",1-5d,Adds very short-horizon pullback behavior dist...,HIGH,already_implemented,Initial diversity expansion roadmap from 01B a...,Short-term overreaction and liquidity-demand s...,Low to moderate; may overlap during selloffs b...,MEDIUM,Keep simple and avoid using future intraday in...
1,wave_1,short_term_reversal,gap_reversal_1d,implemented,Negative overnight gap from prior close to cur...,"open,close",1-5d,Separates overnight dislocation reversal from ...,HIGH,already_implemented,Initial diversity expansion roadmap from 01B a...,Large overnight repricing can overshoot before...,"Moderate in stress, lower in normal regimes.",MEDIUM,"Use only current open and previous close, with..."
2,wave_1,volume_flow,up_down_volume_pressure_20,implemented,Rolling difference between volume on up days a...,"close,volume",5-20d,Adds participation/flow information not captur...,HIGH,already_implemented,Initial diversity expansion roadmap from 01B a...,Persistent accumulation or distribution can re...,Low to moderate; flow pressure can be positive...,LOW,Do not dollar-weight by future prices; use con...
3,wave_1,volume_flow,volume_price_confirmation_20,planned,20-day return multiplied by rolling volume z-s...,"close,volume",10-20d,Tests whether moves supported by unusual volum...,MEDIUM,later_or_low,Initial diversity expansion roadmap from 01B a...,Price moves with participation may be more dur...,Moderate; volume spikes can accompany volatili...,MEDIUM,Clip or rank to reduce domination by single ex...
4,wave_1,liquidity,amihud_illiq_20,implemented,Rolling mean of abs daily return divided by do...,"close,volume",5-20d,Adds tradability/liquidity-premium behavior di...,HIGH,already_implemented,Initial diversity expansion roadmap from 01B a...,Illiquid names may require compensation or exp...,Moderate; related to volatility but scaled by ...,MEDIUM,Handle zero dollar volume safely and leave mis...
5,wave_1,liquidity,dollar_volume_change_20,planned,20-day rolling dollar volume divided by 60-day...,"close,volume",5-20d,Captures changing investor attention and liqui...,LOW,later_or_low,Initial diversity expansion roadmap from 01B a...,Rising trading activity can precede repricing ...,Low to moderate; activity changes need not imp...,LOW,Use trailing averages only.
6,wave_1,correlation_dispersion,market_beta_change_60,implemented,Rolling 20-day beta to benchmark minus rolling...,"close,benchmark_close",10-20d,Identifies changing market sensitivity rather ...,HIGH,already_implemented,Initial diversity expansion roadmap from 01B a...,"Rapid beta changes can reveal crowding, de-ris...",Moderate; beta instability may rise in stress ...,MEDIUM,Use benchmark returns from existing clean benc...
7,wave_1,correlation_dispersion,idiosyncratic_vol_ratio_20_60,planned,20-day residual volatility to benchmark divide...,"close,benchmark_close",5-20d,Separates idiosyncratic shock behavior from br...,MEDIUM,later_or_low,Initial diversity expansion roadmap from 01B a...,Firm-specific risk shocks can mean-revert or p...,"Moderate to high in stress, but more stock-spe...",MEDIUM_HIGH,Treat as diagnostic candidate; 03G should deci...
8,wave_1,volatility_change,volatility_shock_5_20,planned,5-day realized volatility divided by 20-day re...,close,1-10d,Captures volatility acceleration rather than p...,MEDIUM,later_or_low,Initial diversity expansion roadmap from 01B a...,Fresh volatility shocks may behave differently...,Moderate; intentionally related but less level...,MEDIUM_HIGH,Include only if 03G shows it is not a clone of...
9,wave_1,volatility_shock,range

SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,plan,signal_family_expansion_plan_current,signal_family_expansion_plan_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


Final summary


,metric,value
0,run_id,phase2_signal_family_plan_20260502_215958
1,run_timestamp,2026-05-02 21:59:58
2,signal_family_plan_version,phase2_signal_family_plan_v1
3,current_signal_families,16
4,proposed_signal_families,10
5,proposed_signal_count,20
6,expansion_waves,2
7,wave_2_proposed_signal_count,8
8,implemented_wave_1_count,5
